# DESI DR1 AGN Catalogs

Ragadeepika Pucha, Stéphanie Juneau, Arjun Dey, Mar Mezcua, Yao-Yuan Mao, and others ...

In [1]:
import numpy as np
from astropy.table import Table, join, Column, vstack

import sys
sys.path.append('/global/cfs/cdirs/desi/users/raga19/repos/DESI_Project/py/')

import bpt
import utils

from glob import glob

In [2]:
t = Table.read('../catalogs/iron-selection-v2.3.fits')
len(t)

6099572

In [3]:
## VI Flags

vi_out = glob('../02_Analysis/VI_Files/*.txt')

tables = []

for file in vi_out:
    tab = Table.read(file, format = 'ascii')
    tables.append(tab)

t_vi = vstack(tables)
t = join(t, t_vi, keys = ['TARGETID'], join_type = 'left')
t['VI_FLAG'].fill_value = 0
t = t.filled()

In [4]:
## Stellar Mass related variables

logmass = t['LOGM'].data
lomass = (logmass <= 9.5)
himass = (logmass > 9.5)

len(t[lomass]), len(t[himass])

(987987, 5111585)

In [5]:
## Morphology related variables
mtype = t['TYPE'].astype(str).data
ser = t['SERSIC'].data

is_psf = mtype == 'PSF'
is_late = ((mtype == 'REX')|(mtype == 'EXP')|((mtype == 'SER')&(ser <= 2)))
is_early = ~is_psf & ~is_late

In [6]:
## Double-Peaked Emission Line Flags
is_sii_dbl = t['SII_DBL_FLAG'].data
is_oiii_dbl = t['OIII_DBL_FLAG'].data

len(t[is_sii_dbl]), len(t[is_oiii_dbl])

(24735, 15702)

In [7]:
## Emission-line measurements -- Narrow only and sum

## Narrow Only
nii_n = t['NII6583_FLUX'].data
nii_snr_n = nii_n/t['NII6583_FLUX_ERR'].data
nii_aon = t['NII6583_AMPLITUDE'].data/t['NII_HA_NOISE'].data

oiii_n = t['OIII5007_FLUX'].data
oiii_snr_n = oiii_n/t['OIII5007_FLUX_ERR'].data
oiii_aon = t['OIII5007_AMPLITUDE'].data/t['OIII_NOISE'].data

ha_n = t['HA_N_FLUX'].data
ha_snr_n = ha_n/t['HA_N_FLUX_ERR'].data
ha_aon = t['HA_N_AMPLITUDE'].data/t['NII_HA_NOISE'].data

hb_n = t['HB_N_FLUX'].data
hb_snr_n = hb_n/t['HB_N_FLUX_ERR'].data
hb_aon = t['HB_N_AMPLITUDE'].data/t['HB_NOISE']


## Sum
nii_sum = t['NII6583_FLUX'].data + t['NII6583_OUT_FLUX'].data
nii_snr_sum = nii_sum/np.sqrt((t['NII6583_FLUX_ERR'].data**2) + (t['NII6583_OUT_FLUX_ERR'].data**2))

oiii_sum = t['OIII5007_FLUX'].data + t['OIII5007_OUT_FLUX'].data
oiii_snr_sum = oiii_sum/np.sqrt((t['OIII5007_FLUX_ERR'].data**2) + (t['OIII5007_OUT_FLUX_ERR'].data**2))

ha_sum = t['HA_N_FLUX'].data + t['HA_OUT_FLUX'].data
ha_snr_sum = ha_sum/np.sqrt((t['HA_N_FLUX_ERR'].data**2) + (t['HA_OUT_FLUX_ERR'].data**2))

hb_sum = t['HB_N_FLUX'].data + t['HB_OUT_FLUX'].data
hb_snr_sum = hb_sum/np.sqrt((t['HB_N_FLUX_ERR'].data**2) + (t['HB_OUT_FLUX_ERR'].data**2))

In [8]:
## Emission-line measurements

## Empty arrays
nii, nii_snr = np.zeros(len(t)), np.zeros(len(t))
oiii, oiii_snr = np.zeros(len(t)), np.zeros(len(t))
ha, ha_snr = np.zeros(len(t)), np.zeros(len(t))
hb, hb_snr = np.zeros(len(t)), np.zeros(len(t))

## [OIII] Condition
oiii[is_oiii_dbl], oiii[~is_oiii_dbl] = oiii_sum[is_oiii_dbl], oiii_n[~is_oiii_dbl]
oiii_snr[is_oiii_dbl], oiii_snr[~is_oiii_dbl] = oiii_snr_sum[is_oiii_dbl], oiii_snr_n[~is_oiii_dbl]

## [NII]+Ha Condition
nii[is_sii_dbl], nii[~is_sii_dbl] = nii_sum[is_sii_dbl], nii_n[~is_sii_dbl]
nii_snr[is_sii_dbl], nii_snr[~is_sii_dbl] = nii_snr_sum[is_sii_dbl], nii_snr_n[~is_sii_dbl]

ha[is_sii_dbl], ha[~is_sii_dbl] = ha_sum[is_sii_dbl], ha_n[~is_sii_dbl]
ha_snr[is_sii_dbl], ha_snr[~is_sii_dbl] = ha_snr_sum[is_sii_dbl], ha_snr_n[~is_sii_dbl]

hb[is_sii_dbl], hb[~is_sii_dbl] = hb_sum[is_sii_dbl], hb_n[~is_sii_dbl]
hb_snr[is_sii_dbl], hb_snr[~is_sii_dbl] = hb_snr_sum[is_sii_dbl], hb_snr_n[~is_sii_dbl]

nii_ha = np.log10(nii/ha)
oiii_hb = np.log10(oiii/hb)

In [9]:
is_agn, is_comp, is_sf = bpt.Classify_NII_BPT(nii_ha, oiii_hb)

In [10]:
## BL Related Variables

z = t['Z'].data
ha_b = t['HA_B_FLUX'].data
ha_b_err = t['HA_B_FLUX_ERR'].data
ha_b_snr = ha_b/ha_b_err
ha_b_aon = t['HA_B_AMPLITUDE'].data/t['NII_HA_NOISE'].data
ha_b_sig = t['HA_B_SIGMA'].data
ha_b_sig_err = t['HA_B_SIGMA_ERR'].data
ha_b_sig_snr = ha_b_sig/ha_b_sig_err
per_broad = t['PROB_BROAD'].data

ha_b_lum, ha_b_lum_err = utils.Flux_to_Luminosity(ha_b*1e-17, z, ha_b_err*1e-17)
ha_b_fwhm, ha_b_fwhm_err = utils.sigma_to_fwhm(ha_b_sig, ha_b_sig_err)
bh_mass, bh_mass_err1 = utils.calculate_bh_masses(ha_b_lum, ha_b_fwhm, ha_b_lum_err, ha_b_fwhm_err)
bh_mass_err = np.sqrt((bh_mass_err1**2)+(0.5**2))

is_imbh = (bh_mass <= 6)

In [11]:
## Cuts
bpt_cut = (nii_snr >= 3)&(oiii_snr >= 3)&(ha_snr >= 3)&(hb_snr >= 1)&(hb_aon >= 1)
bl_cut = (ha_b_fwhm >= 300)&(ha_b_fwhm < 15000)&(ha_b_snr >= 3)&(ha_b_sig_snr >= 3)&(ha_b_aon >= 2)&(per_broad >= 80)&(t['HA_B_SIGMA'].data > t['HA_N_SIGMA'].data)&(t['HA_N_AMPLITUDE'].data > t['HA_B_AMPLITUDE'].data)

is_bl = bpt_cut&bl_cut
is_nl = bpt_cut&~bl_cut

In [12]:
is_def = (t['NII_HA_SII_NDOF'].data == 0)&(t['HB_OIII_NDOF'].data == 0)
is_ebl = (t['HB_NDOF'].data == 0)&(t['OIII_NDOF'].data == 0)&(t['NII_HA_NDOF'].data == 0)&(t['SII_NDOF'].data == 0)

In [13]:
## VI Flags: Set VI of NL Candidates as -1
t['VI_FLAG'][is_nl] = np.array([-1]*len(t[is_nl]))

fwhm_lt1000 = (ha_b_fwhm < 1000)
vi_flag = t['VI_FLAG'].data
vi0 = vi_flag == 0
vi1 = vi_flag == 1
vi2 = vi_flag == 2
vi3 = vi_flag == 3
vi4 = vi_flag == 4

### Adding Required Columns

In [14]:
## BPT Column

bpt = np.array(['']*len(t), dtype = '<U8')
bpt[is_agn] = 'agn'
bpt[is_comp] = 'comp'
bpt[is_sf] = 'sf'

bpt_col = Column(bpt, name = 'NII_BPT')

In [15]:
## Add EmLine Fluxes column

oiii_col = Column(oiii, name = 'OIII_5007_FLUX')
oiii_snr_col = Column(oiii_snr, name = 'OIII_5007_SNR')
hb_col = Column(hb, name = 'HBETA_FLUX')
hb_snr_col = Column(hb_snr, name = 'HBETA_SNR')
nii_col = Column(nii, name = 'NII_6583_FLUX')
nii_snr_col = Column(nii_snr, name = 'NII_6583_SNR')
ha_col = Column(ha, name = 'HALPHA_FLUX')
ha_snr_col = Column(ha_snr, name = 'HALPHA_SNR')
ha_b_col = Column(ha_b, name = 'HALPHA_BROAD_FLUX')
ha_b_snr_col = Column(ha_b_snr, name = 'HALPHA_BROAD_SNR')
ha_b_lum_col = Column(np.log10(ha_b_lum), name = 'HALPHA_BROAD_LOGLUM')
ha_b_fwhm_col = Column(ha_b_fwhm, name = 'HALPHA_BROAD_FWHM')
bh_mass_col = Column(bh_mass, name = 'LOG_MBH')
bh_mass_err_col = Column(bh_mass_err, name = 'LOG_MBH_ERR')

In [16]:
## Morph-type Column
morph = np.array(['']*len(t), dtype = '<U8')
morph[is_psf] = 'psf'
morph[is_early] = 'early'
morph[is_late] = 'late'

morph_col = Column(morph, name = 'LSDR9_MORPHOLOGY')

In [17]:
## Regular vs. EBL Column

reg_ebl = np.zeros(len(t), dtype = np.bool)
reg_ebl[is_def] = False
reg_ebl[is_ebl] = True

reg_ebl_col = Column(reg_ebl, name = 'EBL_AGN')

In [18]:
t.add_columns([oiii_col, oiii_snr_col, hb_col, hb_snr_col, nii_col, nii_snr_col, ha_col, ha_snr_col, ha_b_col, ha_b_snr_col, ha_b_lum_col, ha_b_fwhm_col, bh_mass_col, bh_mass_err_col, morph_col, bpt_col, reg_ebl_col])

### BPT-AGN Catalogs

In [19]:
sel_columns = ['TARGETID', 'SURVEY', 'PROGRAM', 'HEALPIX', 'TARGET_RA', 'TARGET_DEC', 'Z', \
              'OIII_5007_FLUX', 'OIII_5007_SNR', 'HBETA_FLUX', 'HBETA_SNR', 'NII_6583_FLUX', 'NII_6583_SNR', \
              'HALPHA_FLUX', 'HALPHA_SNR', 'NII_BPT', 'LOGM', 'LOGM_ERR']

t_agn = t[(is_nl|is_bl)&(is_agn|is_comp)][sel_columns]
t_lomass_agn = t[(is_nl|is_bl)&(is_agn|is_comp)&lomass][sel_columns]

In [20]:
# t_agn.write('../catalogs/agn_cats/desi-dr1-all-bpt-agn.fits', overwrite = True)
# t_lomass_agn.write('../catalogs/agn_cats/desi-dr1-all-dwarf-bpt-agn.fits', overwrite = True)

In [21]:
len(t_agn), len(t_agn.colnames)

(323893, 18)

In [22]:
t_agn.colnames

['TARGETID',
 'SURVEY',
 'PROGRAM',
 'HEALPIX',
 'TARGET_RA',
 'TARGET_DEC',
 'Z',
 'OIII_5007_FLUX',
 'OIII_5007_SNR',
 'HBETA_FLUX',
 'HBETA_SNR',
 'NII_6583_FLUX',
 'NII_6583_SNR',
 'HALPHA_FLUX',
 'HALPHA_SNR',
 'NII_BPT',
 'LOGM',
 'LOGM_ERR']

In [23]:
# t1_agn = t_agn['TARGETID','SURVEY','PROGRAM','HEALPIX','TARGET_RA','TARGET_DEC', 'Z'] 
# t1_agn.write('../catalogs/agn_cats/desi-dr1-bpt-agn.fits', overwrite = True)

In [24]:
# t1_lomass_agn = t_lomass_agn['TARGETID','SURVEY','PROGRAM','HEALPIX','TARGET_RA','TARGET_DEC', 'Z'] 
# t1_lomass_agn.write('../catalogs/agn_cats/desi-dr1-dwarf-bpt-agn.fits', overwrite = True)

### BL-AGN Catalogs

In [25]:
## Columns to select

sel_columns = ['TARGETID', 'SURVEY', 'PROGRAM', 'HEALPIX', 'TARGET_RA', 'TARGET_DEC', 'Z', 'LSDR9_MORPHOLOGY', \
              'OIII_5007_FLUX', 'OIII_5007_SNR', 'HBETA_FLUX', 'HBETA_SNR', 'NII_6583_FLUX', 'NII_6583_SNR', \
              'HALPHA_FLUX', 'HALPHA_SNR', 'HALPHA_BROAD_FLUX', 'HALPHA_BROAD_SNR', \
              'HALPHA_BROAD_LOGLUM', 'HALPHA_BROAD_FWHM', 'EBL_AGN', 'NII_BPT', 'LOGM', 'LOGM_ERR', \
               'LOG_MBH', 'LOG_MBH_ERR', 'VI_FLAG']

t_blagn = t[is_bl&(is_agn|is_comp)&(vi0|vi1)][sel_columns]
t_imbh = t[is_bl&(is_agn|is_comp)&(vi0|vi1)&(is_imbh)][sel_columns]

In [26]:
# t_blagn.write('../catalogs/agn_cats/desi-dr1-all-bl-agn.fits', overwrite = True)
# t_imbh.write('../catalogs/agn_cats/desi-dr1-all-imbh.fits', overwrite = True)

In [27]:
len(t_blagn), len(t_blagn.colnames)

(17949, 27)

In [28]:
t_blagn.colnames

['TARGETID',
 'SURVEY',
 'PROGRAM',
 'HEALPIX',
 'TARGET_RA',
 'TARGET_DEC',
 'Z',
 'LSDR9_MORPHOLOGY',
 'OIII_5007_FLUX',
 'OIII_5007_SNR',
 'HBETA_FLUX',
 'HBETA_SNR',
 'NII_6583_FLUX',
 'NII_6583_SNR',
 'HALPHA_FLUX',
 'HALPHA_SNR',
 'HALPHA_BROAD_FLUX',
 'HALPHA_BROAD_SNR',
 'HALPHA_BROAD_LOGLUM',
 'HALPHA_BROAD_FWHM',
 'EBL_AGN',
 'NII_BPT',
 'LOGM',
 'LOGM_ERR',
 'LOG_MBH',
 'LOG_MBH_ERR',
 'VI_FLAG']

In [29]:
# t1_blagn = t_blagn['TARGETID', 'SURVEY', 'PROGRAM', 'HEALPIX', 'TARGET_RA', 'TARGET_DEC', 'Z', \
# 'LSDR9_MORPHOLOGY', 'LOGM', 'LOGM_ERR', 'LOG_MBH', 'LOG_MBH_ERR', 'VI_FLAG']
# t1_blagn.write('../../catalogs/agn_cats/desi-dr1-bl-agn.fits', overwrite = True)

In [30]:
# t1_imbh = t_imbh['TARGETID', 'SURVEY', 'PROGRAM', 'HEALPIX', 'TARGET_RA', 'TARGET_DEC', 'Z', \
# 'LSDR9_MORPHOLOGY', 'LOGM', 'LOGM_ERR', 'LOG_MBH', 'LOG_MBH_ERR', 'VI_FLAG']
# t1_imbh.write('../../catalogs/agn_cats/desi-dr1-imbh.fits', overwrite = True)

---